In [ ]:
import pandas as pd
import os
import json
curr_wd = os.path.abspath(os.getcwd())
print(curr_wd)
from importlib import reload


In [ ]:
fpath = curr_wd + "/data/input/RG_motif_info_df.parquet"
motif_info_set_df = pd.read_parquet(fpath)
motif_info_set_df

fpath = curr_wd + "/data/input/all_IDR_human.parquet"
IDR_info_df = pd.read_parquet(fpath)
IDR_info_df.rename(columns={"protein_name": "UniqueID"}, inplace=True)
IDR_info_df

# from window_extraction import prepare_window_dataframe
import src.window_extraction as window_extraction
# Reload the module after making changes
reload(window_extraction)

# from fasta_writer import build_fasta_records, write_fasta_chunked

df_windows_7 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "7"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)
print(df_windows_7.head())
print(df_windows_7.shape)

df_windows_5 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "5"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_5.head())
print(df_windows_5.shape)

df_windows_4 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "4"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_4.head())
print(df_windows_4.shape)

df_windows_6 = window_extraction.prepare_window_dataframe(
    df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == "6"],
    df_idr=IDR_info_df,
    flank=5,
    mode="adaptive",
    min_idr_fraction=0.9
)

print(df_windows_6.head())
print(df_windows_6.shape)


###### SAVE THE METADATA FOR LATER

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group4_RG_regions_win5_metadata.pkl"

df_windows_4.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group5_RG_regions_win5_metadata.pkl"

df_windows_5.to_pickle(os.path.join(path, filename))

################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group6_RG_regions_win5_metadata.pkl"

df_windows_6.to_pickle(os.path.join(path, filename))
################################################

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group7_RG_regions_win5_metadata.pkl"

df_windows_7.to_pickle(os.path.join(path, filename))


#### GENERATE JSON FILES FOR GNOMAD

result_neg = df_windows_4[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group4_RG_regions_win5.json"


with open(f"{path}/{filename}", "w") as f:
    json.dump(result_neg, f, indent=2)

#######################################

result_pos = df_windows_5[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group5_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

#######################################

result_pos = df_windows_6[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group6_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)


#######################################

result_pos = df_windows_7[["UniqueID", "win_start", "win_end"]].rename(
    columns={"win_start": "start", "win_end": "end"}
).to_dict(orient="records")

path = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
filename = "group7_RG_regions_win5.json"

with open(f"{path}/{filename}", "w") as f:
    json.dump(result_pos, f, indent=2)

In [ ]:
import os
import json
from importlib import reload
import pandas as pd

import src.window_extraction as window_extraction
reload(window_extraction)

# ── Config ────────────────────────────────────────────────────────────────
GROUPS = ["4", "5", "6", "7"]
FLANK = 5
MIN_IDR_FRACTION = 0.9
MODE = "adaptive"

PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
WINDOW_SUFFIX = f"win{FLANK}"

# ── Load inputs ───────────────────────────────────────────────────────────
motif_info_set_df = pd.read_parquet(
    os.path.join(curr_wd, "data/input/RG_motif_info_df.parquet")
)
IDR_info_df = (
    pd.read_parquet(os.path.join(curr_wd, "data/input/all_IDR_human.parquet"))
      .rename(columns={"protein_name": "UniqueID"})
)

# ── Process all groups ────────────────────────────────────────────────────
df_windows_by_group = {}

for group in GROUPS:
    print(f"\n══════════ Group {group} ══════════")
    
    df_windows = window_extraction.prepare_window_dataframe(
        df_motif=motif_info_set_df[motif_info_set_df["Groups_num"] == group],
        df_idr=IDR_info_df,
        flank=FLANK,
        mode=MODE,
        min_idr_fraction=MIN_IDR_FRACTION,
    )
    print(f"  Shape: {df_windows.shape}")
    print(df_windows.head())
    
    df_windows_by_group[group] = df_windows
    
    # Save metadata pickle
    metadata_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}_metadata.pkl"
    )
    df_windows.to_pickle(metadata_path)
    print(f"  Saved metadata: {metadata_path}")
    
    # Save JSON for gnomAD pipeline
    json_records = (
        df_windows[["UniqueID", "win_start", "win_end"]]
        .rename(columns={"win_start": "start", "win_end": "end"})
        .to_dict(orient="records")
    )
    json_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}.json"
    )
    with open(json_path, "w") as f:
        json.dump(json_records, f, indent=2)
    print(f"  Saved JSON: {json_path}")

# ── Summary ───────────────────────────────────────────────────────────────
print("\n══════════ Summary ══════════")
for group, df in df_windows_by_group.items():
    print(f"  Group {group}: {len(df):,} windows from "
          f"{df['UniqueID'].nunique():,} proteins")

In [ ]:
import os
import json
from importlib import reload

import src.gather_genomic_coordinates as gather_genomic_coordinates
import src.write_bed_file as write_bed_file
reload(gather_genomic_coordinates)
reload(write_bed_file)

# ── Config ────────────────────────────────────────────────────────────────
GROUPS = ["4", "5", "6", "7"]
PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
WINDOW_SUFFIX = "win5"

# ── Step 1: gather genomic coordinates per group ──────────────────────────
genomic_coordinates_by_group = {}
failed_by_group = {}

for group in GROUPS:
    print(f"\n══════════ Group {group}: gathering genomic coordinates ══════════")
    
    input_path = os.path.join(
        PROCESSED_DIR, f"group{group}_RG_regions_{WINDOW_SUFFIX}.json"
    )
    regions = gather_genomic_coordinates._read_input(input_path)
    print(f"  Loaded {len(regions)} regions from {input_path}")
    
    genomic_coords, failed = gather_genomic_coordinates.process_multiple_regions(
        regions, parallel=False,
    )
    
    # Add region_id to each entry
    for entry in genomic_coords:
        entry["region_id"] = (
            f"{entry['protein']}_"
            f"{entry['prot_region'][0]}_"
            f"{entry['prot_region'][1]}"
        )
    
    genomic_coordinates_by_group[group] = genomic_coords
    failed_by_group[group] = failed
    
    print(f"  Succeeded: {len(genomic_coords)}, Failed: {len(failed)}")
    if failed:
        print(f"  Failed regions: {failed}")

# ── Step 2: merge all groups into one JSON ────────────────────────────────
merged_genomic_coordinates_list = []
for group in GROUPS:
    merged_genomic_coordinates_list.extend([
        {**d, "group": group}
        for d in genomic_coordinates_by_group[group]
    ])

print(f"\nMerged total: {len(merged_genomic_coordinates_list)} regions "
      f"across {len(GROUPS)} groups")

merged_json_path = os.path.join(
    PROCESSED_DIR, f"genomic_coords_merged_{WINDOW_SUFFIX}_4groups.json"
)
with open(merged_json_path, "w") as fh:
    json.dump(merged_genomic_coordinates_list, fh, indent=4)
print(f"Saved merged JSON: {merged_json_path}")

# ── Step 3: write combined BED file ───────────────────────────────────────
bed_path = os.path.join(
    PROCESSED_DIR, f"genomic_coords_combined_{WINDOW_SUFFIX}_4groups.bed"
)
write_bed_file.write_multigroup_bed(
    results_by_group=genomic_coordinates_by_group,
    output_path=bed_path,
)

In [ ]:
import src.variant_assignment as va

# Load
df_raw = va.load_vep_tsv(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/combined_vep_variants_4groups.tsv"
)

print("══ Basic structure ══")
print(f"Total rows: {len(df_raw):,}")
print(f"Columns: {df_raw.columns.tolist()}")

print("\n══ Quality distributions ══")
print("\nFILTER:")
print(df_raw["FILTER"].value_counts().head(5))
print("\nBIOTYPE:")
print(df_raw["BIOTYPE"].value_counts().head(5))
print("\nConsequence (top 10):")
print(df_raw["Consequence"].value_counts().head(10))

print("\n══ Chromosome coverage ══")
print("\nRows per chromosome (should match BED counts roughly):")
print(df_raw["CHROM"].value_counts().sort_index())

print("\n══ Identity check: protein coverage ══")
# How many unique UniProt accessions appear in the VEP output?
df_raw["uniprot_short"] = df_raw["UNIPROT_ISOFORM"].apply(
    lambda x: str(x).split("-")[0] if pd.notna(x) else None
)
n_unique_uniprot = df_raw["uniprot_short"].nunique()
n_unique_symbol = df_raw["SYMBOL"].nunique()
print(f"Unique UniProt accessions in VEP output: {n_unique_uniprot}")
print(f"Unique gene symbols in VEP output: {n_unique_symbol}")

print("\n══ Cross-check: how many of our queried proteins were found? ══")
# Load any one of the window metadata pickles to get expected proteins
import os
PROCESSED_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
all_expected_proteins = set()
for group in ["4", "5", "6", "7"]:
    df_windows = pd.read_pickle(
        os.path.join(PROCESSED_DIR, f"group{group}_RG_regions_win5_metadata.pkl")
    )
    all_expected_proteins.update(df_windows["UniqueID"].unique())
print(f"Expected proteins (across all 4 groups): {len(all_expected_proteins)}")

found_proteins = set(df_raw["uniprot_short"].dropna().unique())
overlap = all_expected_proteins & found_proteins
missing_from_vep = all_expected_proteins - found_proteins
print(f"Found in VEP output: {len(overlap)} / {len(all_expected_proteins)}")
print(f"Missing from VEP output: {len(missing_from_vep)}")

if len(missing_from_vep) > 0:
    print(f"\nFirst 20 missing proteins:")
    for p in list(missing_from_vep)[:20]:
        print(f"  {p}")

print("\n══ Filtering effect (apply filter) ══")
df_filt = va.filter_vep(df_raw)
print(f"After filtering: {len(df_filt):,} rows ({100*len(df_filt)/len(df_raw):.1f}% kept)")

In [ ]:
# How complete is UniProt in the filtered set?
print("Rows with UNIPROT_ISOFORM populated:", df_filt["UNIPROT_ISOFORM"].notna().sum())
print("Rows without:", df_filt["UNIPROT_ISOFORM"].isna().sum())
print()
print("Sample of populated values:")
print(df_filt["UNIPROT_ISOFORM"].dropna().sample(10).tolist())
print()
print("Gene symbols where UNIPROT_ISOFORM is missing:")
print(df_filt[df_filt["UNIPROT_ISOFORM"].isna()]["SYMBOL"].value_counts().head(10))

import json

with open("/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5_4groups.json") as f:
    regions = json.load(f)

uniprot_ids = sorted({r["protein"] for r in regions})
print(f"Unique UniProt IDs in regions: {len(uniprot_ids)}")
print(uniprot_ids[:10])

import src.variant_assignment as va

uniprot_to_symbols = va.fetch_uniprot_to_symbols(uniprot_ids)

# Check coverage
print(f"Got mappings for {len(uniprot_to_symbols)} / {len(uniprot_ids)} UniProt IDs")

# Any missing?
missing = set(uniprot_ids) - set(uniprot_to_symbols.keys())
if missing:
    print(f"Missing: {missing}")

# Sanity check — does RBMXL2 appear in any reverse mapping?
for acc, symbols in uniprot_to_symbols.items():
    if "RBMXL2" in symbols:
        print(f"RBMXL2 → UniProt {acc}")
        break


uniprot_lookup, symbol_lookup = va.build_region_lookups(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/genomic_coords_merged_win5_4groups.json",
    uniprot_to_symbols,
)

df_assigned = va.assign_variants_to_regions(df_filt, uniprot_lookup, symbol_lookup)
df_assigned.head()

In [ ]:
# How many of your 781 regions got at least one variant?
region_coverage = df_assigned.groupby("region_id").size()
n_regions_with_variants = len(region_coverage)
print(f"Regions with at least one variant: {n_regions_with_variants} / 781")
print(f"Regions with no variants: {781 - n_regions_with_variants}")

print("\nVariants per region distribution:")
print(region_coverage.describe())

# Per-group coverage
print("\nVariants per group:")
print(df_assigned.groupby("group").size())



In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Inference data processing: parse AAs → AlphaMissense → save final parquet
# ════════════════════════════════════════════════════════════════════════════
import json
import pandas as pd

# ── Step 1: Parse amino acids ─────────────────────────────────────────────
df_parsed = va.parse_amino_acids_column(df_assigned)

# Sanity check across consequence types
sample_cons = ["missense_variant", "synonymous_variant", "stop_gained",
               "inframe_deletion", "inframe_insertion", "start_lost"]
for cons in sample_cons:
    subset = df_parsed[df_parsed["Consequence"] == cons]
    if len(subset) > 0:
        print(f"\n{cons}:")
        print(subset[["Amino_acids", "before_aa", "after_aa"]].head(3).to_string(index=False))
print(f"\nNulls in before_aa: {df_parsed['before_aa'].isna().sum()}")
print(f"Nulls in after_aa: {df_parsed['after_aa'].isna().sum()}")

# ── Step 2: Extract UniProt IDs for AlphaMissense filtering ───────────────
with open(
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/"
    "genomic_coords_merged_win5_4groups.json"
) as f:
    regions = json.load(f)
uniprot_ids = sorted({r["protein"] for r in regions})
print(f"\nUniProt IDs in inference regions: {len(uniprot_ids)}")

# ── Step 3: Load AlphaMissense (or reuse cached if available) ─────────────
# CHANGED: new cached filename for inference
am_cache_path = (
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/"
    "am_filtered_for_4groups_inference.parquet"
)

try:
    am = pd.read_parquet(am_cache_path)
    print(f"Loaded cached AlphaMissense: {len(am):,} rows")
except FileNotFoundError:
    print("Cache not found, loading from full AlphaMissense file...")
    am = va.load_alphamissense_for_proteins(
        am_tsv_path=(
            "/mnt/d/phd/scripts/16_ev_signature_predictor/data/alphamissense/"
            "AlphaMissense_aa_substitutions.tsv.gz"
        ),
        uniprot_ids=uniprot_ids,
    )
    am.to_parquet(am_cache_path)
    print(f"Saved cache: {am_cache_path}")

# ── Step 4: Merge AlphaMissense onto variants ─────────────────────────────
df_final = va.merge_alphamissense(df_parsed, am)

# Missing-protein check
missing_from_am = set(uniprot_ids) - set(am["uniprot_id"])
print(f"\nUniProt IDs missing from AlphaMissense: {len(missing_from_am)}")
if missing_from_am and len(missing_from_am) <= 20:
    print(f"  Missing: {missing_from_am}")

# ── Step 5: Per-group descriptive stats (no pos/neg comparison) ───────────
missense = df_final[df_final["Consequence"].str.contains("missense_variant", na=False)]
print("\nMissense pathogenicity by group:")
print(missense.groupby("group")["am_pathogenicity"].describe())
print("\nAlphaMissense class breakdown by group:")
print(missense.groupby(["group", "am_class"]).size().unstack(fill_value=0))

print("\nGene counts per group:")
for group in sorted(df_final["group"].unique()):
    n_genes = df_final[df_final["group"] == group]["SYMBOL"].nunique()
    n_regions = df_final[df_final["group"] == group]["region_id"].nunique()
    print(f"  group {group}: {n_genes} genes, {n_regions} regions")

# ── Step 6: Save final annotated variants parquet ─────────────────────────
# CHANGED: new filename to avoid overwriting training data
output_path = (
    "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed/"
    "variants_annotated_final_4groups_inference.parquet"
)
df_final.to_parquet(output_path)
print(f"\nSaved {len(df_final):,} rows × {len(df_final.columns)} columns")
print(f"  → {output_path}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Inference: variant annotation → RG annotation → features → predictions
# ════════════════════════════════════════════════════════════════════════════
%load_ext autoreload
%autoreload 2

import sys, os
import json
import numpy as np
import pandas as pd

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"

# ── Step 1: Load the inference variants + regions ─────────────────────────
df = pd.read_parquet(
    f"{DATA_DIR}/variants_annotated_final_4groups_inference.parquet"
)
print(f"Loaded {len(df):,} variant-region assignments")

with open(f"{DATA_DIR}/genomic_coords_merged_win5_4groups.json") as f:
    regions = json.load(f)
region_by_id = {r["region_id"]: r for r in regions}
print(f"Loaded {len(regions)} regions from JSON")

# ── Step 2: Coordinate convention fix ─────────────────────────────────────
# Identify half-open regions (need patching) and broken regions (need dropping)
halfopen_region_ids = []
broken_region_ids = []
for r in regions:
    start, end = r["prot_region"]
    seq_len = len(r["prot_seq"])
    if seq_len < 2:
        broken_region_ids.append(r["region_id"])
    elif seq_len == end - start:
        halfopen_region_ids.append(r["region_id"])

print(f"Half-open regions to patch: {len(halfopen_region_ids)}")
print(f"Broken regions to drop: {len(broken_region_ids)}")

# Apply patches
for rid in halfopen_region_ids:
    r = region_by_id[rid]
    old_start, old_end = r["prot_region"]
    r["prot_region"] = [old_start + 1, old_end]
    assert len(r["prot_seq"]) == r["prot_region"][1] - r["prot_region"][0] + 1, \
        f"Patch failed for {rid}"

for rid in broken_region_ids:
    del region_by_id[rid]

# Flag and filter variants
df["coord_patched"] = df["region_id"].isin(halfopen_region_ids)
df["coord_dropped"] = df["region_id"].isin(broken_region_ids)
df = df[~df["coord_dropped"]].copy()
print(f"After dropping broken regions: {len(df):,} variants")

patched_mask = df["region_id"].isin(halfopen_region_ids)
df.loc[patched_mask, "region_start_aa"] = df.loc[patched_mask, "region_start_aa"] + 1

# ── Step 3: WT match check ────────────────────────────────────────────────
def _check(row):
    region = region_by_id.get(row["region_id"])
    if region is None or pd.isna(row["protein_position_int"]):
        return None
    pos = int(row["protein_position_int"]) - int(row["region_start_aa"])
    if pos < 0 or pos >= len(region["prot_seq"]):
        return None
    if row["before_aa"] is None:
        return None
    if row["before_aa"] == "-" or len(row["before_aa"]) != 1:
        return None
    return region["prot_seq"][pos] == row["before_aa"]

df["wt_match"] = df.apply(_check, axis=1)
df["rg_analysis_reliable"] = df["wt_match"] == True

df_for_rg = df[df["rg_analysis_reliable"]].copy()
print(f"\nWT match summary:")
print(df["wt_match"].value_counts(dropna=False))
print(f"\nUsable for RG analysis: {len(df_for_rg):,}")
print(f"\nBreakdown by group:")
print(df_for_rg["group"].value_counts())

# ── Step 4: RG annotation ─────────────────────────────────────────────────
import src.analysis_visualization.rg_analysis as rga

df_rg = rga.compute_rg_disruption_columns(df_for_rg, region_by_id)
print(f"\nRG annotation done. Summary:")
print(df_rg["hits_rg"].value_counts())

# ── Step 5: RG event classification ───────────────────────────────────────
df_events = rga.compute_rg_change_events(df_rg, region_by_id)
print(f"\nRG events computed. Summary:")
print(df_events["rg_change_event"].value_counts(dropna=False))

# ── Step 6: Physchem deltas ───────────────────────────────────────────────
import src.analysis_visualization.physchem_analysis as pca
physchem_deltas_df = pca.compute_physchem_deltas(df_rg, region_by_id)
print(f"\nPhyschem deltas computed: {len(physchem_deltas_df):,} variants")

# ── Step 7: Build features for the classifier ─────────────────────────────
import src.analysis_visualization.classifier_features as cf

features_df = cf.build_classifier_features(
    df_rg=df_rg,
    df_events=df_events,
    region_by_id=region_by_id,
    physchem_deltas_df=physchem_deltas_df,
    dataset="gnomad_inference",
)

# Save features for safety
features_path = f"{DATA_DIR}/classifier_features_4groups_inference.parquet"
features_df.to_parquet(features_path)
print(f"\nSaved features: {features_path}")
print(f"Shape: {features_df.shape}")
print(f"\nGroup distribution:")
print(features_df["group"].value_counts())

In [ ]:
"""
Classifier inference + SHAP explanations.

Uses the full-feature classifier (all features including AlphaMissense)
to score regions and explain each prediction.

Output structure:
    protein_id  motif_start  motif_end  win_start  win_end  region_seq
    group  score  top_3_pushing_pos  top_3_pushing_neg  region_id
"""

import os
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path


# ── Config ────────────────────────────────────────────────────────────────
DATA_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/processed"
MODELS_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/models"
OUTPUT_DIR = "/mnt/d/phd/scripts/16_ev_signature_predictor/data/output"

Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

CLASSIFIER_FILE = "classifier_all_features.joblib"


# ════════════════════════════════════════════════════════════════════════════
# Step 1: Load features_df (output of build_classifier_features)
# ════════════════════════════════════════════════════════════════════════════
features_path = os.path.join(
    DATA_DIR, "classifier_features_4groups_inference.parquet"
)
features_df = pd.read_parquet(features_path)
print(f"Loaded features_df: {features_df.shape}")
print(f"Groups: {features_df['group'].value_counts().to_dict()}")


# ════════════════════════════════════════════════════════════════════════════
# Step 2: Load the all-features classifier
# ════════════════════════════════════════════════════════════════════════════
classifier_path = os.path.join(MODELS_DIR, CLASSIFIER_FILE)
bundle = joblib.load(classifier_path)
print(f"\nLoaded classifier: {bundle['label']}, "
      f"{bundle['n_features']} features, "
      f"trained on {bundle['n_train_samples']} samples")

model = bundle["model"]
imputer = bundle["imputer"]
expected_features = bundle["feature_names"]


# ════════════════════════════════════════════════════════════════════════════
# Step 3: Align inference features to the classifier's training schema
# ════════════════════════════════════════════════════════════════════════════
inference_cols = set(features_df.columns)
expected_cols = set(expected_features)
missing = expected_cols - inference_cols
extra = inference_cols - expected_cols - {"region_id", "group", "label"}

if missing:
    print(f"\nWARNING: {len(missing)} features expected by classifier "
          f"are missing in inference data:")
    for f in sorted(missing)[:20]:
        print(f"  {f}")
    if len(missing) > 20:
        print(f"  ... and {len(missing) - 20} more")
    # Add missing features as all-NaN columns (imputer will fill them)
    for f in missing:
        features_df[f] = np.nan

if extra:
    print(f"\nNote: {len(extra)} extra features in inference data "
          f"will be ignored by classifier")

# Extract features in TRAINING ORDER and impute
X = features_df[expected_features].values
X_imputed = imputer.transform(X)


# ════════════════════════════════════════════════════════════════════════════
# Step 4: Predict scores
# ════════════════════════════════════════════════════════════════════════════
print("\nPredicting scores...")
y_proba = model.predict_proba(X_imputed)[:, 1]
features_df["score"] = y_proba
print(f"  → Scored {len(features_df)} regions")


# ════════════════════════════════════════════════════════════════════════════
# Step 5: Compute SHAP values (explanation per region)
# ════════════════════════════════════════════════════════════════════════════
print("\nBuilding SHAP TreeExplainer...")
explainer = shap.TreeExplainer(model)

print("Computing SHAP values...")
shap_values_raw = explainer.shap_values(X_imputed)

# Handle different SHAP API versions
if isinstance(shap_values_raw, list):
    # Old API: list of [neg_class_shap, pos_class_shap]
    shap_values = shap_values_raw[1]
elif shap_values_raw.ndim == 3:
    # New API: 3D array (samples, features, classes)
    shap_values = shap_values_raw[:, :, 1]
else:
    shap_values = shap_values_raw

# Baseline (expected value) for positive class
expected_value_raw = explainer.expected_value
if isinstance(expected_value_raw, (list, np.ndarray)) and \
   hasattr(expected_value_raw, '__len__') and len(expected_value_raw) == 2:
    baseline = expected_value_raw[1]
else:
    baseline = expected_value_raw

print(f"SHAP values shape: {shap_values.shape}")
print(f"Baseline (expected value of pos class): {baseline:.4f}")

# Sanity check
sample_idx = 0
predicted_score = features_df["score"].iloc[sample_idx]
shap_sum = shap_values[sample_idx].sum() + baseline
print(f"\nSanity check on row {sample_idx}:")
print(f"  Predicted score: {predicted_score:.4f}")
print(f"  Baseline + SHAP sum: {shap_sum:.4f}")


# ════════════════════════════════════════════════════════════════════════════
# Step 6: Save full SHAP matrix (for power users / deep dives)
# ════════════════════════════════════════════════════════════════════════════
shap_df = pd.DataFrame(
    shap_values,
    columns=expected_features,
    index=features_df["region_id"].values,
)
shap_df.index.name = "region_id"
shap_df["__baseline"] = baseline

shap_path = os.path.join(OUTPUT_DIR, "shap_values_4groups_inference.parquet")
shap_df.to_parquet(shap_path)
print(f"\nSaved full SHAP matrix: {shap_path}")


# ════════════════════════════════════════════════════════════════════════════
# Step 7: Compute per-region top features (3 positive + 3 negative)
# ════════════════════════════════════════════════════════════════════════════

def format_top_features(shap_row, feature_names, n_top=3, direction="positive"):
    """
    Top N features pushing the score in the specified direction,
    formatted as a readable string.
    """
    paired = list(zip(feature_names, shap_row))
    if direction == "positive":
        top = sorted(paired, key=lambda x: -x[1])[:n_top]
    else:
        top = sorted(paired, key=lambda x: x[1])[:n_top]
    return ", ".join([f"{name} ({val:+.3f})" for name, val in top])


features_df["top_3_pushing_pos"] = [
    format_top_features(shap_values[i], expected_features, n_top=3,
                         direction="positive")
    for i in range(len(shap_values))
]
features_df["top_3_pushing_neg"] = [
    format_top_features(shap_values[i], expected_features, n_top=3,
                         direction="negative")
    for i in range(len(shap_values))
]


# ════════════════════════════════════════════════════════════════════════════
# Step 8: Enrich with metadata (motif coords, region sequence)
# ════════════════════════════════════════════════════════════════════════════
# Load region info (has prot_seq and prot_region)
with open(os.path.join(DATA_DIR, "genomic_coords_merged_win5_4groups.json")) as f:
    regions = json.load(f)
region_lookup = {r["region_id"]: r for r in regions}

# Load window metadata for motif coords
window_metadata_combined = []
for group in ["4", "5", "6", "7"]:
    df_w = pd.read_pickle(
        os.path.join(DATA_DIR, f"group{group}_RG_regions_win5_metadata.pkl")
    )
    df_w["group"] = group
    window_metadata_combined.append(df_w)
window_meta = pd.concat(window_metadata_combined, ignore_index=True)

# Parse region_id format "UniProtID_win_start_win_end"
def parse_region_id(rid):
    parts = rid.rsplit("_", 2)
    if len(parts) == 3:
        return parts[0], int(parts[1]), int(parts[2])
    return None, None, None

features_df["UniqueID_parsed"] = features_df["region_id"].apply(
    lambda r: parse_region_id(r)[0]
)
features_df["win_start_parsed"] = features_df["region_id"].apply(
    lambda r: parse_region_id(r)[1]
)
features_df["win_end_parsed"] = features_df["region_id"].apply(
    lambda r: parse_region_id(r)[2]
)

# Merge motif coords from window metadata
metadata_subset = window_meta[[
    "UniqueID", "win_start", "win_end", "motif_start", "motif_end"
]].drop_duplicates()

features_enriched = features_df.merge(
    metadata_subset,
    left_on=["UniqueID_parsed", "win_start_parsed", "win_end_parsed"],
    right_on=["UniqueID", "win_start", "win_end"],
    how="left",
)

# Add region_seq
features_enriched["region_seq"] = features_enriched["region_id"].apply(
    lambda r: region_lookup.get(r, {}).get("prot_seq", "")
)


# ════════════════════════════════════════════════════════════════════════════
# Step 9: Build final output table
# ════════════════════════════════════════════════════════════════════════════
output_cols = [
    "UniqueID",
    "motif_start",
    "motif_end",
    "win_start",
    "win_end",
    "region_seq",
    "group",
    "score",
    "top_3_pushing_pos",
    "top_3_pushing_neg",
    "region_id",
]
final_table = features_enriched[output_cols].copy()
final_table = final_table.rename(columns={"UniqueID": "protein_id"})
final_table = final_table.sort_values(
    ["group", "score"], ascending=[True, False]
).reset_index(drop=True)
final_table["score"] = final_table["score"].round(4)


# ════════════════════════════════════════════════════════════════════════════
# Step 10: Save and report
# ════════════════════════════════════════════════════════════════════════════
parquet_path = os.path.join(OUTPUT_DIR, "predictions_4groups_inference.parquet")
tsv_path = os.path.join(OUTPUT_DIR, "predictions_4groups_inference.tsv")
final_table.to_parquet(parquet_path)
final_table.to_csv(tsv_path, sep="\t", index=False)

print(f"\n══ Saved predictions ══")
print(f"Parquet: {parquet_path}")
print(f"TSV:     {tsv_path}")
print(f"SHAP matrix: {shap_path}")
print(f"Shape: {final_table.shape}")

# Score distributions per group
print(f"\n══ Score distributions by group ══")
for group in sorted(final_table["group"].unique()):
    sub = final_table[final_table["group"] == group]
    print(f"\nGroup {group} (n={len(sub)}):")
    print(f"  median = {sub['score'].median():.3f}, "
          f"mean = {sub['score'].mean():.3f}")
    print(f"  ≥ 0.5: {(sub['score'] >= 0.5).sum()}")
    print(f"  ≥ 0.7: {(sub['score'] >= 0.7).sum()}")
    print(f"  ≥ 0.9: {(sub['score'] >= 0.9).sum()}")

# Top 10 with SHAP explanations
print(f"\n══ Top 10 scoring regions with SHAP explanations ══")
top10 = final_table.nlargest(10, "score")
for _, row in top10.iterrows():
    print(f"\n{row['protein_id']} ({row['motif_start']}-{row['motif_end']}) "
          f"— group {row['group']}")
    print(f"  Score: {row['score']:.3f}")
    print(f"  Sequence: {row['region_seq']}")
    print(f"  Pushed UP by: {row['top_3_pushing_pos']}")
    print(f"  Pushed DOWN by: {row['top_3_pushing_neg']}")

print(f"\n══ Bottom 5 scoring regions ══")
bot5 = final_table.nsmallest(5, "score")
for _, row in bot5.iterrows():
    print(f"\n{row['protein_id']} ({row['motif_start']}-{row['motif_end']}) "
          f"— group {row['group']}")
    print(f"  Score: {row['score']:.3f}")
    print(f"  Pushed UP by: {row['top_3_pushing_pos']}")
    print(f"  Pushed DOWN by: {row['top_3_pushing_neg']}")